# Business Demography: struttura delle imprese piccole

Questo foglio usa Eurostat Business Demography per osservare le classi dimensionali fini disponibili fino a 9 dipendenti. Il dataset descrive distribuzioni di imprese e occupazione, non valore aggiunto.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Base dati

            La fonte copre una finestra storica piu lunga rispetto al dataset SBS recente e arriva al 2020 per il dataset configurato.

In [ ]:
bd_raw = read_project_csv("data/raw/business_demography_raw.csv")
bd = enrich_business_demography(bd_raw)

pd.DataFrame(
    {
        "righe": [len(bd)],
        "paesi": [bd["country_code"].nunique()],
        "anni": [f'{int(bd["year"].min())}-{int(bd["year"].max())}'],
        "settori": [bd["sector_code_original"].nunique()],
        "classi": [bd["size_label"].nunique()],
        "metriche": [bd["metric_code"].nunique()],
    }
)

## Imprese fino a 9 dipendenti per classe

Business Demography viene usata qui solo per le micro-classi 0, 1-4 e 5-9 dipendenti. La classe aggregata 10+ e esclusa per non mescolare un blocco residuale con classi molto piu fini.

In [ ]:
latest_year = latest_year_with_values(bd, metric_code="V11910")
active = (
    bd.query("metric_code == 'V11910' and year == @latest_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
active["size_label"] = pd.Categorical(active["size_label"], SIZE_ORDER_BUSINESS, ordered=True)
active_share = add_share(active, ["country_label"])

fig = px.bar(
    active_share.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_BUSINESS},
    labels={"country_label": "Paese", "share_pct": "% imprese fino a 9 dipendenti", "size_label": "Classe"},
    title=f"Distribuzione delle imprese fino a 9 dipendenti per classe - {latest_year}",
)
add_source_note(fig, "Eurostat Business Demography")
fig

## Italia nel tempo

Per l'Italia, la serie storica evidenzia se la composizione delle micro-classi sotto i 10 dipendenti cambia nel tempo o resta stabile.

In [ ]:
it_trend = (
    bd.query("metric_code == 'V11910' and country_code == 'IT'")
    .dropna(subset=["value"])
    .groupby(["year", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
it_trend["size_label"] = pd.Categorical(it_trend["size_label"], SIZE_ORDER_BUSINESS, ordered=True)
it_trend = add_share(it_trend, ["year"])
quota_per_anno = it_trend.groupby("year", observed=True)["share_pct"].sum().round(6)
assert quota_per_anno.between(99.999, 100.001).all(), quota_per_anno

fig = px.area(
    it_trend.sort_values(["year", "size_label"]),
    x="year",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_BUSINESS},
    labels={"year": "Anno", "share_pct": "% imprese fino a 9 dipendenti", "size_label": "Classe"},
    title="Italia: composizione delle imprese fino a 9 dipendenti per classe dimensionale",
)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.update_layout(plot_bgcolor="white")
add_source_note(fig, "Eurostat Business Demography")
fig

## Dove sono le classi sopra i 10 dipendenti

Le classi sopra i 10 dipendenti non sono sparite: stanno in Structural Business Statistics, dove e disponibile il valore aggiunto per classe dimensionale. Business Demography resta utile solo per aprire il blocco sotto i 10 dipendenti in micro-classi descrittive.

In [ ]:
perimetro_classi = pd.DataFrame(
    [
        {
            "Fonte": "Business Demography",
            "Uso nel progetto": "micro-classi descrittive sotto i 10 dipendenti",
            "Classi mostrate": ", ".join(SIZE_ORDER_BUSINESS),
            "Valore aggiunto": "No",
        },
        {
            "Fonte": "Structural Business Statistics",
            "Uso nel progetto": "valore aggiunto per dimensione d'impresa",
            "Classi mostrate": ", ".join(SIZE_ORDER_OFFICIAL),
            "Valore aggiunto": "Si",
        },
    ]
)
perimetro_classi

## Sintesi tabellare

La tabella ordina i paesi per peso della classe 0 dipendenti sul totale delle micro-classi osservate.

In [ ]:
(
    active_share.pivot_table(
        index="country_label",
        columns="size_label",
        values="share_pct",
        aggfunc="sum",
        observed=True,
    )
    .reindex(columns=SIZE_ORDER_BUSINESS)
    .round(1)
    .sort_values(SIZE_ORDER_BUSINESS[0], ascending=False)
)